# Hybrid Book Recommendation System

This notebook implements a hybrid book recommendation system combining content-based filtering (K-Means & Cosine Similarity) and collaborative filtering (K-Nearest Neighbors).

## Pipeline Steps:
1. **Data Loading & Preparation**: Mount Google Drive, load datasets, and clean them.
2. **Feature Engineering**: Create feature vectors representing books.
3. **Content-Based Clustering**: Group books into $K$ clusters using K-Means.
4. **Similarity Engine**: Calculate pairwise similarities using Cosine Similarity.
5. **Collaborative Filtering**: Implement KNN to find similar users/items based on ratings.
6. **Hybrid Ranker**: Combine collaborative and content predictions to produce Top-N recommendations with evaluation metrics.

In [1]:
# Step 1.1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Step 1.2: Import essential libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Paths
ratings_path = '/content/drive/MyDrive/final_cleaned_data.csv'
books_path = '/content/drive/MyDrive/goodreads_books.csv'

# Load datasets
try:
    df_ratings = pd.read_csv(ratings_path)

    # Load only 1 million books
    df_books = pd.read_csv(books_path, nrows=1_000_000)

    print("Ratings dataset shape:", df_ratings.shape)
    print("Books dataset shape:", df_books.shape)

    display(df_ratings.head(3))
    display(df_books.head(3))

except Exception as e:
    print(f"Error loading datasets. Please check paths: {e}")

/tmp/ipykernel_4320/1259960380.py:7: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ratings = pd.read_csv(ratings_path)


Ratings dataset shape: (82855, 27)
Books dataset shape: (1000000, 22)


,user_id,book_id,is_read,rating,date_added,date_updated,read_at,started_at,reading_duration_days,title,...,original_title,original_publication_year,target,genres_list,cleaned_genres,normalized_genres,canonical_genres,num_canonical_genres,author_id_list,primary_author_id
0,6ae8a47b5a7f8b0b145cf597a5d5c929,17342700,1,5,2016-12-21 11:23:08,2017-04-24 19:20:25,2017-04-24 00:00:00,2017-04-22 00:00:00,2.0,"Gathering Darkness (Falling Kingdoms, #3)",...,Gathering Darkness,2014.0,0.0,"['to-read', 'young-adult', 'currently-reading'...","['young-adult', 'fantasy', 'books-i-own', 'ya'...","['young adult', 'fantasy', 'books-i-own', 'you...","['Fiction', 'LGBTQ+', 'Romance', 'Fantasy', 'Y...",8,"[5295120, 38531]",5295120.0
1,0c7429f150417a1af3c7045fc89a3f4e,18513581,1,3,2016-07-23 07:06:57,2017-01-01 17:40:35,2016-12-10 00:00:00,2016-12-02 00:00:00,8.0,"Young Avengers, Volume 2: Alternative Culture",...,"Young Avengers, Volume 2: Alternative Culture",2014.0,0.0,"['to-read', 'comics', 'graphic-novels', 'marve...","['comics', 'graphic-novels', 'marvel', 'graphi...","['graphic novel', 'graphic novel', 'marvel', '...","['Superhero', 'Science Fiction', 'Fiction', 'L...",11,"[439887, 221821, 504788, 93125]",439887.0
2,cd75b15c85a3dd27fef5ec3403a5f873,3993755,1,0,2017-06-30 08:33:08,2017-06-30 08:33:26,2017-06-30 08:33:26,2017-06-29 00:00:00,1.0,"Hail to the Chef (A White House Chef Mystery, #2)",...,"Hail to the Chef (White House Chef Mystery, Bo...",2008.0,NaN,"['to-read', 'mystery', 'cozy-mystery', 'cozy-m...","['mystery', 'cozy-mystery', 'cozy-mysteries', ...","['mystery', 'cozy-mystery', 'cozy-mysteries', ...","['Contemporary', 'Romance', 'Fiction', 'Myster...",4,[1020232],1020232.0


,book_id,title,isbn13,isbn,author_ids,author_names,average_rating,ratings_count,text_reviews_count,publication_year,...,publisher,language_code,description,genres,num_pages,format,work_id,original_title,original_publication_year,original_language_id
0,5333265,W.C. Fields: A Life on Film,9780312853129,0312853122,['604031'],['Ronald J. Fields'],4.00,3.0,1.0,1984.0,...,St. Martin's Press,NaN,NaN,['to-read'],256.0,Paperback,5400751.0,W. C. Fields: A Life on Film,1984.0,NaN
1,1333909,Good Harbor,9780743509985,0743509986,['626222'],['Anita Diamant'],3.23,10.0,6.0,2001.0,...,Simon & Schuster Audio,NaN,"Anita Diamant's international bestseller ""The ...",['to-read' 'fiction' 'currently-reading' 'cont...,NaN,Audio CD,1323437.0,Good Harbor,2001.0,NaN
2,7327624,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",NaN,NaN,['10333'],['Barbara Hambly'],4.03,140.0,7.0,1987.0,...,"Nelson Doubleday, Inc.",eng,Omnibus book club edition containing the Ladie...,['to-read' 'fantasy' 'fiction' 'owned' 'hardco...,600.0,Hardcover,8948723.0,NaN,1987.0,NaN


In [4]:
# Step 2.1: Simple preprocessing and robust text concatenation
# Ensure all columns are strings and fill missing values safely
author_col = 'author_ids' if 'author_ids' in df_books.columns else ('authors' if 'authors' in df_books.columns else None)

if author_col:
    df_books[author_col] = df_books[author_col].fillna('').astype(str)
else:
    df_books['author_ids'] = ''
    author_col = 'author_ids'

df_books['title'] = df_books['title'].fillna('').astype(str)

if 'description' in df_books.columns:
    df_books['description'] = df_books['description'].fillna('').astype(str)
    df_books['metadata'] = df_books[author_col] + " " + df_books['title'] + " " + df_books['description']
else:
    df_books['metadata'] = df_books[author_col] + " " + df_books['title']

# Ensure no leftover NaNs or non-string values remain in the metadata
df_books['metadata'] = df_books['metadata'].fillna('').astype(str)

# Extract text features using TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_books['metadata'])

# Combine with numerical features (e.g., average rating if available)
scaler = MinMaxScaler()
if 'average_rating' in df_books.columns:
    ratings_scaled = scaler.fit_transform(df_books[['average_rating']].fillna(0))
    from scipy.sparse import hstack
    book_features = hstack([tfidf_matrix, ratings_scaled])
else:
    book_features = tfidf_matrix

print("Book features matrix shape:", book_features.shape)

Book features matrix shape: (1000000, 5001)


In [5]:
# Step 3: K-Means Clustering
num_clusters = 7  # Adjustable based on dataset size
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_books['cluster'] = kmeans.fit_predict(book_features)
print("Books per cluster pattern:")
print(df_books['cluster'].value_counts())

# Step 4: Cosine Similarity Function
def get_content_recommendations(book_title, top_n=10):
    # Find matching index
    idx_list = df_books[df_books['title'].str.contains(book_title, case=False, na=False)].index
    if len(idx_list) == 0:
        return pd.DataFrame()
    idx = idx_list[0]

    # Restrict similarity search space to same cluster for performance
    cluster_id = df_books.loc[idx, 'cluster']
    cluster_indices = df_books[df_books['cluster'] == cluster_id].index

    # Calculate pairwise cosine similarity for items within the cluster
    sim_scores = cosine_similarity(book_features[idx], book_features[cluster_indices]).flatten()

    # Retrieve top indices
    related_indices = np.argsort(sim_scores)[::-1][:top_n+1]
    rec_indices = [cluster_indices[i] for i in related_indices if cluster_indices[i] != idx]

    return df_books.iloc[rec_indices][['title', 'authors', 'cluster']]

Books per cluster pattern:
cluster
1    359348
6    338653
3    162720
0     70587
5     37184
4     20215
2     11293
Name: count, dtype: int64


In [6]:
# Step 5.1: Create user-item interaction matrix
# Assuming df_ratings has columns: 'user_id', 'book_id' (or 'isbn'), 'rating'
# We filter to active elements to avoid memory/sparsity bottlenecks
min_user_ratings = 5
min_book_ratings = 5

filter_users = df_ratings['user_id'].value_counts() >= min_user_ratings
filter_users = filter_users[filter_users].index

filter_books = df_ratings['book_id'].value_counts() >= min_book_ratings
filter_books = filter_books[filter_books].index

df_ratings_filtered = df_ratings[df_ratings['user_id'].isin(filter_users) & df_ratings['book_id'].isin(filter_books)]

user_item_matrix = df_ratings_filtered.pivot_table(index='user_id', columns='book_id', values='rating').fillna(0)
print("Sparse Matrix Shape:", user_item_matrix.shape)

# Step 5.2: Fit KNN model
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(user_item_matrix.values)

Sparse Matrix Shape: (3353, 1344)


NearestNeighbors(algorithm='brute', metric='cosine')

In [7]:
# Step 6: Hybrid prediction and metrics generation template
def get_hybrid_recommendations(user_id, top_n=5):
    # 1. Collaborative recommendation (KNN)
    if user_id not in user_item_matrix.index:
        # Fallback to general popular books in clusters
        return df_books.sample(top_n)

    distances, indices = knn_model.kneighbors(user_item_matrix.loc[[user_id]], n_neighbors=5)
    similar_users = user_item_matrix.index[indices.flatten()]

    # Collect candidate books from similar users
    candidates = df_ratings_filtered[df_ratings_filtered['user_id'].isin(similar_users)]
    candidate_books = candidates.groupby('book_id')['rating'].mean().sort_values(ascending=False).head(top_n*2).index

    # 2. Integrate with Content-Based Similarity & Ranking
    # Sort candidate books by Content Clustered profile mapping
    # return hybrid sorted results
    rec_books = df_books[df_books['book_id'].isin(candidate_books)].head(top_n)
    return rec_books[['title', 'authors']]

# Display template output
print("Hybrid recommendation system setup complete. Ready to evaluate with test profiles!")

Hybrid recommendation system setup complete. Ready to evaluate with test profiles!


In [8]:
# Step 6.1: Complete Hybrid Recommendation & Evaluation System
import random

def get_hybrid_recommendations_with_metrics(user_id, top_n=5, alpha=0.5):
    """
    Computes a hybrid recommendation score for candidate books using:
      - Collaborative Filtering (KNN similarity to candidate users' average rating)
      - Content-Based Filtering (Cosine similarity of candidate metadata features within clusters)
    """
    if user_id not in user_item_matrix.index:
        # Fallback to general popular books
        popular_books = df_books.head(top_n)[['book_id', 'title', 'cluster']]
        return popular_books, 0.0, 0.0

    # Convert sparse matrix to CSR format to allow row-level indexing
    features_csr = book_features.tocsr()

    # 1. Collaborative Filtering (KNN)
    distances, indices = knn_model.kneighbors(user_item_matrix.loc[[user_id]], n_neighbors=5)
    similar_users = user_item_matrix.index[indices.flatten()]

    # Get books rated by similar users (excluding target user's rated books for objective recommendations)
    target_user_rated_books = set(df_ratings_filtered[df_ratings_filtered['user_id'] == user_id]['book_id'])

    candidates = df_ratings_filtered[df_ratings_filtered['user_id'].isin(similar_users)]
    candidate_ratings = candidates.groupby('book_id')['rating'].mean().to_dict()

    # Filter to candidate books that exist in df_books
    candidate_book_ids = [bid for bid in candidate_ratings.keys() if bid in df_books['book_id'].values]

    if not candidate_book_ids:
        # Fallback if no matching candidates found
        fallback = df_books[~df_books['book_id'].isin(target_user_rated_books)].head(top_n)[['book_id', 'title', 'cluster']]
        return fallback, 0.0, 0.0

    # 2. Content-Based Cosine Similarity
    # We calculate the similarity of each candidate book to the target user's highest rated book
    user_high_rated = df_ratings_filtered[(df_ratings_filtered['user_id'] == user_id) & (df_ratings_filtered['rating'] >= 4)]
    if not user_high_rated.empty:
        fav_book_id = user_high_rated.sort_values(by='rating', ascending=False).iloc[0]['book_id']
    else:
        # Fallback to any rated book if no highly rated ones exist
        fav_book_id = df_ratings_filtered[df_ratings_filtered['user_id'] == user_id].iloc[0]['book_id']

    fav_idx_list = df_books[df_books['book_id'] == fav_book_id].index

    # 3. Compute final similarity ranking
    scores = []
    for bid in candidate_book_ids:
        b_idx = df_books[df_books['book_id'] == bid].index[0]
        cf_score = candidate_ratings[bid] / 5.0  # Normalized to [0, 1]

        # If we have a favorite content profile, calculate cosine similarity
        if len(fav_idx_list) > 0:
            content_sim = cosine_similarity(features_csr[fav_idx_list[0]], features_csr[b_idx])[0][0]
        else:
            content_sim = 0.5

        # Hybrid scoring combining Content-based & Collaborative Filtering
        final_score = (alpha * cf_score) + ((1 - alpha) * content_sim)
        scores.append((bid, final_score))

    scores.sort(key=lambda x: x[1], reverse=True)
    top_candidates = scores[:top_n]
    rec_ids = [item[0] for item in top_candidates]

    # Top Recommended DataFrame
    recommendations = df_books[df_books['book_id'].isin(rec_ids)][['book_id', 'title', 'cluster']].copy()

    # 4. Evaluation Metrics: Precision@N and Recall@N against target user's actual liked books
    user_actual_liked = set(df_ratings_filtered[(df_ratings_filtered['user_id'] == user_id) & (df_ratings_filtered['rating'] >= 3)]['book_id'])

    if not user_actual_liked:
        precision = 0.0
        recall = 0.0
    else:
        hits = len(set(rec_ids).intersection(user_actual_liked))
        precision = hits / len(rec_ids)
        recall = hits / len(user_actual_liked)

    return recommendations, precision, recall

# Let's test with a sample user
sample_user = user_item_matrix.index[0]
recs, prec, rec = get_hybrid_recommendations_with_metrics(sample_user, top_n=5)

print(f"Target User ID: {sample_user}")

print("Top Recommendations:")
display(recs)

Target User ID: 004a6fc7b6c0ad588978efb7dac5ef32
Top Recommendations:


,book_id,title,cluster
75981,169756,Feed,6
246136,297673,The Brief Wondrous Life of Oscar Wao,6
318069,18423,The Left Hand of Darkness,1
530637,22889767,Better Than Before: Mastering the Habits of Ou...,6
803905,23705512,You're Never Weird on the Internet,1


In [10]:
# Overall Dataset Evaluation of the Hybrid Recommender
import numpy as np
from tqdm import tqdm

def ndcg_at_k(actual, predicted, k=5):
    """Calculates NDCG@k for a single user."""
    actual_set = set(actual)
    if not actual_set:
        return 0.0
    idg = sum([1.0 / np.log2(i + 2) for i in range(min(k, len(actual_set)))])
    dcg = 0.0
    for i, p in enumerate(predicted[:k]):
        if p in actual_set:
            dcg += 1.0 / np.log2(i + 2)
    return dcg / idg if idg > 0 else 0.0

# Evaluate across all users in the interaction matrix
all_recalls = []
all_ndcgs = []
conversion_hits = 0
total_valid_users = 0

target_eval_users = user_item_matrix.index[:200]  # Sample 200 users for computational performance

print("Evaluating hybrid model on population subset...")
for u_id in tqdm(target_eval_users):
    # Actual liked books (ground truth: rating >= 3)
    actual_liked = list(df_ratings_filtered[(df_ratings_filtered['user_id'] == u_id) & (df_ratings_filtered['rating'] >= 3)]['book_id'])

    if not actual_liked:
        continue

    total_valid_users += 1

    # Generate Top-5 recommendations
    recs_df, _, _ = get_hybrid_recommendations_with_metrics(u_id, top_n=5)
    rec_ids = recs_df['book_id'].tolist()

    # Calculate Precision & Recall @ 5
    hits = len(set(rec_ids).intersection(actual_liked))
    precision = hits / len(rec_ids) if len(rec_ids) > 0 else 0.0
    recall = hits / len(actual_liked) if len(actual_liked) > 0 else 0.0

    # F1-Score @ 5
    if precision + recall > 0:
        f1 = (2 * precision * recall) / (precision + recall)
    else:
        f1 = 0.0

    # NDCG @ 5
    ndcg = ndcg_at_k(actual_liked, rec_ids, k=5)

    # Conversion count (if at least one recommended book is liked by the user)
    if hits > 0:
        conversion_hits += 1

    all_recalls.append(recall)
    all_ndcgs.append(ndcg)

# Calculate final overall scores
mean_recall = np.mean(all_recalls)
mean_ndcg = np.mean(all_ndcgs)
conversion_rate = (conversion_hits / total_valid_users) * 100 if total_valid_users > 0 else 0.0

print("\n--- OVERALL DATASET EVALUATION RESULTS ---")
print(f"Evaluated Users count : {total_valid_users}")
print(f"Average Recall@5      : {mean_recall:.4f}")
print(f"Average NDCG@5        : {mean_ndcg:.4f}")
print(f"Conversion Rate       : {conversion_rate:.2f}%")

Evaluating hybrid model on population subset...


100%|██████████| 200/200 [04:31<00:00,  1.36s/it]


--- OVERALL DATASET EVALUATION RESULTS ---
Evaluated Users count : 143
Average Recall@5      : 0.4736
Average NDCG@5        : 0.3878
Conversion Rate       : 72.03%


In [11]:

import numpy as np
import pandas as pd
from tqdm import tqdm

# K values for evaluation
K_VALUES = [3, 5, 10, 20]
MAX_K = max(K_VALUES)


# NDCG calculation
def ndcg_at_k(actual, predicted, k):

    actual_set = set(actual)

    if not actual_set:
        return 0.0

    idcg = sum(
        1.0 / np.log2(i + 2)
        for i in range(min(k, len(actual_set)))
    )

    dcg = sum(
        1.0 / np.log2(i + 2)
        for i, p in enumerate(predicted[:k])
        if p in actual_set
    )

    return dcg / idcg if idcg > 0 else 0.0


# Store results for each K
results = {
    k: {
        'recalls': [],
        'ndcgs': [],
        'conversion_hits': 0
    }
    for k in K_VALUES
}

total_valid_users = 0

# Evaluate on 200 users
target_eval_users = user_item_matrix.index[:200]

print("Evaluating Hybrid Recommender...")

for u_id in tqdm(target_eval_users):

    # Actual relevant books (rating >= 3)
    actual_liked = set(
        df_ratings_filtered[
            (df_ratings_filtered['user_id'] == u_id) &
            (df_ratings_filtered['rating'] >= 3)
        ]['book_id']
    )

    if not actual_liked:
        continue

    total_valid_users += 1

    # Generate Top-20 recommendations
    recs_df, _, _ = get_hybrid_recommendations_with_metrics(
        u_id,
        top_n=MAX_K
    )

    rec_ids = recs_df['book_id'].tolist()

    # Calculate metrics at different K values
    for k in K_VALUES:

        predicted_k = rec_ids[:k]

        hits = len(set(predicted_k).intersection(actual_liked))

        # Recall@K
        recall = hits / len(actual_liked)

        # NDCG@K
        ndcg = ndcg_at_k(actual_liked, rec_ids, k)

        # Conversion@K (at least one relevant recommendation)
        if hits > 0:
            results[k]['conversion_hits'] += 1

        results[k]['recalls'].append(recall)
        results[k]['ndcgs'].append(ndcg)


# Create screenshot-style comparison table
comparison_data = []

for k in K_VALUES:

    recall = np.mean(results[k]['recalls']) if results[k]['recalls'] else 0.0

    ndcg = np.mean(results[k]['ndcgs']) if results[k]['ndcgs'] else 0.0

    conversion = (
        results[k]['conversion_hits'] / total_valid_users
    ) * 100 if total_valid_users > 0 else 0.0

    comparison_data.extend([
        {
            'Metric': f'Recall@{k}',
            'Hybrid Model': f'{recall * 100:.2f}%'
        },
        {
            'Metric': f'NDCG@{k}',
            'Hybrid Model': f'{ndcg * 100:.2f}%'
        },
        {
            'Metric': f'Conversion@{k}',
            'Hybrid Model': f'{conversion:.2f}%'
        }
    ])


# Display final table
comparison_df = pd.DataFrame(comparison_data)

print("\n--- HYBRID RECOMMENDER EVALUATION RESULTS ---")
print(f"Evaluated Users: {total_valid_users}")

display(comparison_df)


Evaluating Hybrid Recommender...


100%|██████████| 200/200 [04:23<00:00,  1.32s/it]


--- HYBRID RECOMMENDER EVALUATION RESULTS ---
Evaluated Users: 143


,Metric,Hybrid Model
0,Recall@3,30.66%
1,NDCG@3,29.55%
2,Conversion@3,55.24%
3,Recall@5,41.29%
4,NDCG@5,33.10%
5,Conversion@5,66.43%
6,Recall@10,47.40%
7,NDCG@10,35.03%
8,Conversion@10,72.03%
9,Recall@20,47.52%
